In [14]:
# ========================================================================
# XGBOOST MODEL: STEAM GAME REVENUE PREDICTION (REVISED)
# ========================================================================
# Business Questions:
# 1. What genres will generate the most revenue?
# 2. Will discounts affect sales/revenue?
#
# Feedback Implemented:
# 1. Include zero-revenue games
# 2. Genre-specific RMSE analysis
# 3. Grid search hyperparameter tuning
# 4. Flask API for model serving
# 5. SHAP analysis for discount effects
#
# Author: [Your Name]
# Date: December 2025
# ========================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("=" * 70)
print("XGBOOST MODEL: PREDICTING STEAM GAME REVENUE (REVISED)")
print("=" * 70)

# ========================================================================
# 1. LOAD DATA AND CREATE REVENUE PROXY
# ========================================================================

print("\n" + "=" * 70)
print("STEP 1: LOADING DATA")
print("=" * 70)

# Load cleaned dataset
df = pd.read_csv('steam_games_cleaned.csv')

print(f"\n✓ Dataset loaded: {df.shape[0]:,} games, {df.shape[1]} columns")

# Create revenue proxy (target variable)
REVIEW_TO_OWNER_RATIO = 75
F2P_REVENUE_PER_PLAYER = 50

df['estimated_owners'] = df['overall_review_count'] * REVIEW_TO_OWNER_RATIO
df['estimated_revenue'] = df['estimated_owners'] * df['final_price']

# F2P games special calculation
df.loc[df['is_free_to_play'] == True, 'estimated_revenue'] = (
    df.loc[df['is_free_to_play'] == True, 'estimated_owners'] * F2P_REVENUE_PER_PLAYER
)

# CHANGE 1: Fill NaN revenue with 0 instead of dropping
df['estimated_revenue'] = df['estimated_revenue'].fillna(0)

print(f"\n✓ Revenue proxy created")
print(f"Total estimated market: ₹{df['estimated_revenue'].sum()/1e9:.2f} Billion")

XGBOOST MODEL: PREDICTING STEAM GAME REVENUE (REVISED)

STEP 1: LOADING DATA

✓ Dataset loaded: 42,497 games, 35 columns

✓ Revenue proxy created
Total estimated market: ₹6004.71 Billion


In [15]:
# ========================================================================
# 2. FEATURE SELECTION (PRE-LAUNCH ONLY)
# ========================================================================

print("\n" + "=" * 70)
print("STEP 2: FEATURE SELECTION")
print("=" * 70)

# Define features
FEATURES = [
    'main_genre',
    'final_price',
    'discount_pct_clean',
    'is_on_sale',
    'win_support',
    'mac_support',
    'linux_support',
    'multi_platform',
    'has_dlc',
    'dlc_available',
    'is_early_access',
    'is_free_to_play',
]

TARGET = 'estimated_revenue'

print(f"\n✓ Selected {len(FEATURES)} features for pre-launch prediction")


STEP 2: FEATURE SELECTION

✓ Selected 12 features for pre-launch prediction


In [16]:
# ========================================================================
# 3. DATA PREPARATION (INCLUDE ZERO REVENUE)
# ========================================================================

print("\n" + "=" * 70)
print("STEP 3: DATA PREPARATION")
print("=" * 70)

# Filter to complete cases
df_model = df[FEATURES + [TARGET]].copy()

# Remove rows with missing FEATURES (but keep zero revenue)
print(f"\nBefore cleaning: {len(df_model):,} games")
df_model = df_model.dropna(subset=FEATURES)  # Only drop if features are missing
print(f"After cleaning: {len(df_model):,} games")

# CHANGE 1: Keep zero-revenue games
zero_revenue_count = (df_model[TARGET] == 0).sum()
nonzero_revenue_count = (df_model[TARGET] > 0).sum()

print(f"\n📊 Revenue Distribution:")
print(f"  Zero revenue games: {zero_revenue_count:,} ({zero_revenue_count/len(df_model)*100:.1f}%)")
print(f"  Non-zero revenue games: {nonzero_revenue_count:,} ({nonzero_revenue_count/len(df_model)*100:.1f}%)")
print(f"  ✓ Keeping ALL games (including zeros) for realistic predictions")


STEP 3: DATA PREPARATION

Before cleaning: 42,497 games
After cleaning: 4,844 games

📊 Revenue Distribution:
  Zero revenue games: 527 (10.9%)
  Non-zero revenue games: 4,317 (89.1%)
  ✓ Keeping ALL games (including zeros) for realistic predictions


In [17]:
# ========================================================================
# 4. FEATURE ENGINEERING
# ========================================================================

print("\n" + "=" * 70)
print("STEP 4: FEATURE ENGINEERING")
print("=" * 70)

# One-hot encode categorical features
genre_dummies = pd.get_dummies(df_model['main_genre'], prefix='genre', drop_first=False)
print(f"✓ Created {len(genre_dummies.columns)} genre features")

# Drop original categorical column and add dummies
df_model = df_model.drop('main_genre', axis=1)
df_model = pd.concat([df_model, genre_dummies], axis=1)

# Convert boolean columns to int
bool_columns = ['win_support', 'mac_support', 'linux_support',
                'multi_platform', 'has_dlc', 'is_early_access',
                'is_free_to_play', 'is_on_sale']

for col in bool_columns:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype(int)

print(f"\n✓ Feature engineering complete")
print(f"Final feature count: {len(df_model.columns) - 1}")


STEP 4: FEATURE ENGINEERING
✓ Created 11 genre features

✓ Feature engineering complete
Final feature count: 22


In [18]:
# ========================================================================
# 5. TRAIN-TEST SPLIT
# ========================================================================

print("\n" + "=" * 70)
print("STEP 5: TRAIN-TEST SPLIT")
print("=" * 70)

# Separate features and target
X = df_model.drop(TARGET, axis=1)
y = df_model[TARGET]

# Log-transform target (handles zeros: log1p(0) = 0)
y_log = np.log1p(y)
print("\n✓ Applied log1p transformation (handles zero values)")

# Split data (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log,
    test_size=0.2,
    random_state=42,
    stratify=None  # Can't stratify continuous target
)

print(f"\n✓ Data split complete:")
print(f"  Training set: {len(X_train):,} games ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Test set: {len(X_test):,} games ({len(X_test)/len(X)*100:.1f}%)")

# Store genre labels for later analysis
# Extract main genre from one-hot encoded columns
genre_cols = [col for col in X.columns if col.startswith('genre_')]
X_train_with_genre = X_train.copy()
X_test_with_genre = X_test.copy()

# Get genre name for each sample
def get_genre_from_onehot(row, genre_cols):
    for col in genre_cols:
        if row[col] == 1:
            return col.replace('genre_', '')
    return 'unknown'

train_genres = X_train[genre_cols].apply(lambda row: get_genre_from_onehot(row, genre_cols), axis=1)
test_genres = X_test[genre_cols].apply(lambda row: get_genre_from_onehot(row, genre_cols), axis=1)


STEP 5: TRAIN-TEST SPLIT

✓ Applied log1p transformation (handles zero values)

✓ Data split complete:
  Training set: 3,875 games (80.0%)
  Test set: 969 games (20.0%)


In [19]:
# ========================================================================
# SAVE FEATURE NAMES FOR API
# ========================================================================

import json

# Get the exact feature names used in training
feature_names = X_train.columns.tolist()

# Save to JSON file
with open('feature_names.json', 'w') as f:
    json.dump(feature_names, f, indent=2)

print(f"✓ Saved {len(feature_names)} feature names")
print("\nFeature names:")
for i, name in enumerate(feature_names):
    print(f"  {i+1}. {name}")

# Download the file
from google.colab import files
files.download('feature_names.json')

✓ Saved 22 feature names

Feature names:
  1. final_price
  2. discount_pct_clean
  3. is_on_sale
  4. win_support
  5. mac_support
  6. linux_support
  7. multi_platform
  8. has_dlc
  9. dlc_available
  10. is_early_access
  11. is_free_to_play
  12. genre_action
  13. genre_adventure
  14. genre_casual
  15. genre_early access
  16. genre_indie
  17. genre_massively multiplayer
  18. genre_racing
  19. genre_rpg
  20. genre_simulation
  21. genre_sports
  22. genre_strategy


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
# ========================================================================
# 6. BASELINE MODEL (Ridge Regression)
# ========================================================================

print("\n" + "=" * 70)
print("STEP 6: BASELINE MODEL (Ridge Regression)")
print("=" * 70)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

baseline_model = Ridge(alpha=1.0)
baseline_model.fit(X_train_scaled, y_train)
y_pred_baseline = baseline_model.predict(X_test_scaled)

baseline_r2 = r2_score(y_test, y_pred_baseline)
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline))

print(f"\n📊 Baseline Model Performance (Ridge Regression):")
print(f"  R² Score: {baseline_r2:.4f}")
print(f"  MAE (log scale): {baseline_mae:.4f}")
print(f"  RMSE (log scale): {baseline_rmse:.4f}")


STEP 6: BASELINE MODEL (Ridge Regression)

📊 Baseline Model Performance (Ridge Regression):
  R² Score: 0.2184
  MAE (log scale): 3.0643
  RMSE (log scale): 4.4353


In [21]:
# ========================================================================
# 7. XGBOOST MODEL - INITIAL TRAINING
# ========================================================================

print("\n" + "=" * 70)
print("STEP 7: XGBOOST MODEL - INITIAL TRAINING")
print("=" * 70)

# Initial model (before tuning)
xgb_initial = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror',
    eval_metric='rmse',
    early_stopping_rounds=50
)

print("\n🚀 Training initial XGBoost model...")
xgb_initial.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

# Evaluate initial model
y_pred_initial = xgb_initial.predict(X_test)
initial_r2 = r2_score(y_test, y_pred_initial)
initial_rmse = np.sqrt(mean_squared_error(y_test, y_pred_initial))

print(f"\n✓ Initial model trained")
print(f"  Test R²: {initial_r2:.4f}")
print(f"  Test RMSE (log): {initial_rmse:.4f}")


STEP 7: XGBOOST MODEL - INITIAL TRAINING

🚀 Training initial XGBoost model...

✓ Initial model trained
  Test R²: 0.3275
  Test RMSE (log): 4.1141


In [22]:
# ========================================================================
# 8. GRID SEARCH HYPERPARAMETER TUNING (CHANGE 3)
# ========================================================================

print("\n" + "=" * 70)
print("STEP 8: GRID SEARCH HYPERPARAMETER TUNING")
print("=" * 70)

print("\n🔍 Starting grid search for optimal hyperparameters...")
print("This may take 5-10 minutes...\n")

# Define parameter grid
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 300],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

# Base model for grid search
xgb_base = xgb.XGBRegressor(
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror',
    eval_metric='rmse'
)

# Grid search with cross-validation
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='r2',
    cv=3,  # 3-fold CV (faster than 5)
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\n✓ Grid search complete!")
print(f"\n📊 Best Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV R² Score: {grid_search.best_score_:.4f}")

# Train final model with best parameters
xgb_model = grid_search.best_estimator_

# Evaluate tuned model
y_pred_test = xgb_model.predict(X_test)
y_pred_train = xgb_model.predict(X_train)

tuned_r2_train = r2_score(y_train, y_pred_train)
tuned_r2_test = r2_score(y_test, y_pred_test)
tuned_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
tuned_mae = mean_absolute_error(y_test, y_pred_test)

# Convert back from log scale
y_test_original = np.expm1(y_test)
y_pred_test_original = np.expm1(y_pred_test)
tuned_rmse_original = np.sqrt(mean_squared_error(y_test_original, y_pred_test_original))
tuned_mae_original = mean_absolute_error(y_test_original, y_pred_test_original)

print(f"\n📊 Tuned Model Performance:")
print(f"  Training R²: {tuned_r2_train:.4f}")
print(f"  Test R²: {tuned_r2_test:.4f}")
print(f"  Train-Test Gap: {tuned_r2_train - tuned_r2_test:.4f}")
print(f"\n  Test RMSE (log): {tuned_rmse:.4f}")
print(f"  Test RMSE (₹): ₹{tuned_rmse_original/1e6:.2f} Million")
print(f"  Test MAE (₹): ₹{tuned_mae_original/1e6:.2f} Million")

# Model comparison
print("\n" + "-" * 70)
print("MODEL COMPARISON:")
print("-" * 70)
print(f"{'Model':<25} {'R² Score':<15} {'RMSE (₹M)':<15} {'Improvement'}")
print("-" * 70)
print(f"{'Ridge Baseline':<25} {baseline_r2:>8.4f}")
print(f"{'XGBoost (Initial)':<25} {initial_r2:>8.4f}       {(initial_r2-baseline_r2)/baseline_r2*100:>+6.1f}%")
print(f"{'XGBoost (Tuned)':<25} {tuned_r2_test:>8.4f}      ₹{tuned_rmse_original/1e6:>6.1f}M      {(tuned_r2_test-initial_r2)/initial_r2*100:>+6.1f}%")
print("-" * 70)


STEP 8: GRID SEARCH HYPERPARAMETER TUNING

🔍 Starting grid search for optimal hyperparameters...
This may take 5-10 minutes...

Fitting 3 folds for each of 486 candidates, totalling 1458 fits

✓ Grid search complete!

📊 Best Parameters:
  colsample_bytree: 0.9
  learning_rate: 0.01
  max_depth: 6
  min_child_weight: 5
  n_estimators: 300
  subsample: 0.7

Best CV R² Score: 0.2347

📊 Tuned Model Performance:
  Training R²: 0.3614
  Test R²: 0.3135
  Train-Test Gap: 0.0480

  Test RMSE (log): 4.1569
  Test RMSE (₹): ₹1001.89 Million
  Test MAE (₹): ₹103.17 Million

----------------------------------------------------------------------
MODEL COMPARISON:
----------------------------------------------------------------------
Model                     R² Score        RMSE (₹M)       Improvement
----------------------------------------------------------------------
Ridge Baseline              0.2184
XGBoost (Initial)           0.3275        +50.0%
XGBoost (Tuned)             0.3135      ₹100

In [23]:
# ========================================================================
# 9. GENRE-SPECIFIC RMSE ANALYSIS (CHANGE 2)
# ========================================================================

print("\n" + "=" * 70)
print("STEP 9: GENRE-SPECIFIC RMSE ANALYSIS")
print("=" * 70)

print("\n📊 WHAT IS RMSE?")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("RMSE = Root Mean Squared Error")
print("• Measures average prediction error in the same units as target (₹)")
print("• Lower RMSE = Better predictions")
print("• Penalizes large errors more than small errors")
print("\nInterpretation:")
print("  RMSE = ₹50M → Predictions are off by ~₹50M on average")
print("  High RMSE → Genre is hard to predict (high revenue variance)")
print("  Low RMSE → Genre has consistent, predictable revenue patterns")

# Calculate RMSE by genre
print("\n" + "-" * 70)
print("GENRE-SPECIFIC PERFORMANCE:")
print("-" * 70)

genre_performance = pd.DataFrame({
    'genre': test_genres,
    'actual': y_test_original,
    'predicted': y_pred_test_original,
    'error': y_test_original - y_pred_test_original,
    'squared_error': (y_test_original - y_pred_test_original)**2
})

genre_rmse = genre_performance.groupby('genre').agg({
    'squared_error': lambda x: np.sqrt(x.mean()),
    'actual': ['count', 'mean', 'std'],
    'error': 'mean'
}).round(2)

genre_rmse.columns = ['RMSE', 'Sample_Count', 'Mean_Revenue', 'Std_Revenue', 'Mean_Error']
genre_rmse['RMSE_millions'] = (genre_rmse['RMSE'] / 1e6).round(2)
genre_rmse['Mean_Revenue_M'] = (genre_rmse['Mean_Revenue'] / 1e6).round(2)
genre_rmse['Std_Revenue_M'] = (genre_rmse['Std_Revenue'] / 1e6).round(2)
genre_rmse['Coefficient_of_Variation'] = (genre_rmse['Std_Revenue'] / genre_rmse['Mean_Revenue']).round(2)

# Sort by RMSE (hardest to predict first)
genre_rmse_sorted = genre_rmse.sort_values('RMSE', ascending=False)

print("\nTop 15 Genres by Prediction Difficulty (RMSE):\n")
print(genre_rmse_sorted[['Sample_Count', 'RMSE_millions', 'Mean_Revenue_M',
                         'Std_Revenue_M', 'Coefficient_of_Variation']].head(15).to_string())

# Analysis
print("\n" + "=" * 70)
print("GENRE PREDICTABILITY ANALYSIS:")
print("=" * 70)

hardest_genre = genre_rmse_sorted.index[0]
easiest_genre = genre_rmse_sorted.index[-1]

print(f"\n🔴 HARDEST TO PREDICT: {hardest_genre.upper()}")
print(f"   RMSE: ₹{genre_rmse_sorted.loc[hardest_genre, 'RMSE_millions']:.1f}M")
print(f"   Mean Revenue: ₹{genre_rmse_sorted.loc[hardest_genre, 'Mean_Revenue_M']:.1f}M")
print(f"   Std Deviation: ₹{genre_rmse_sorted.loc[hardest_genre, 'Std_Revenue_M']:.1f}M")
print(f"   Coefficient of Variation: {genre_rmse_sorted.loc[hardest_genre, 'Coefficient_of_Variation']:.2f}")
print(f"\n   WHY? High variance in revenue (some hits, many flops)")

print(f"\n🟢 EASIEST TO PREDICT: {easiest_genre.upper()}")
print(f"   RMSE: ₹{genre_rmse_sorted.loc[easiest_genre, 'RMSE_millions']:.1f}M")
print(f"   Mean Revenue: ₹{genre_rmse_sorted.loc[easiest_genre, 'Mean_Revenue_M']:.1f}M")
print(f"   Std Deviation: ₹{genre_rmse_sorted.loc[easiest_genre, 'Std_Revenue_M']:.1f}M")
print(f"   Coefficient of Variation: {genre_rmse_sorted.loc[easiest_genre, 'Coefficient_of_Variation']:.2f}")
print(f"\n   WHY? Consistent revenue patterns (low variance)")

print("\n💡 KEY INSIGHT:")
print("   Genres with high Coefficient of Variation are harder to predict")
print("   → Action/RPG may have both mega-hits and flops")
print("   → Niche genres may have more consistent (lower) revenue")


STEP 9: GENRE-SPECIFIC RMSE ANALYSIS

📊 WHAT IS RMSE?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RMSE = Root Mean Squared Error
• Measures average prediction error in the same units as target (₹)
• Lower RMSE = Better predictions
• Penalizes large errors more than small errors

Interpretation:
  RMSE = ₹50M → Predictions are off by ~₹50M on average
  High RMSE → Genre is hard to predict (high revenue variance)
  Low RMSE → Genre has consistent, predictable revenue patterns

----------------------------------------------------------------------
GENRE-SPECIFIC PERFORMANCE:
----------------------------------------------------------------------

Top 15 Genres by Prediction Difficulty (RMSE):

            Sample_Count  RMSE_millions  Mean_Revenue_M  Std_Revenue_M  Coefficient_of_Variation
genre                                                                                           
rpg                   19        5209.15         1461.40        5159.43                 

In [24]:
# ========================================================================
# 10. FEATURE IMPORTANCE ANALYSIS
# ========================================================================

print("\n" + "=" * 70)
print("STEP 10: FEATURE IMPORTANCE ANALYSIS")
print("=" * 70)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 Top 20 Most Important Features:\n")
print(feature_importance.head(20).to_string(index=False))

# Category importance
genre_features = feature_importance[feature_importance['feature'].str.startswith('genre_')]
pricing_features = ['final_price', 'discount_pct_clean', 'is_on_sale']
pricing_importance = feature_importance[feature_importance['feature'].isin(pricing_features)]['importance'].sum()

genre_importance = genre_features['importance'].sum()

print("\n" + "-" * 70)
print("Feature Importance by Category:")
print("-" * 70)
print(f"  Genre features: {genre_importance:.4f} ({genre_importance*100:.1f}%)")
print(f"  Pricing & Discount: {pricing_importance:.4f} ({pricing_importance*100:.1f}%)")


STEP 10: FEATURE IMPORTANCE ANALYSIS

📊 Top 20 Most Important Features:

           feature  importance
           has_dlc    0.433448
     dlc_available    0.059938
       mac_support    0.054277
       final_price    0.053510
         genre_rpg    0.049956
  genre_simulation    0.048420
discount_pct_clean    0.046675
      genre_casual    0.045716
    multi_platform    0.033699
   is_early_access    0.032420
    genre_strategy    0.029974
       genre_indie    0.027621
     linux_support    0.026716
   genre_adventure    0.026625
      genre_action    0.023010
      genre_racing    0.007996
        is_on_sale    0.000000
       win_support    0.000000
   is_free_to_play    0.000000
genre_early access    0.000000

----------------------------------------------------------------------
Feature Importance by Category:
----------------------------------------------------------------------
  Genre features: 0.2593 (25.9%)
  Pricing & Discount: 0.1002 (10.0%)


In [25]:
# ========================================================================
# 11. SHAP ANALYSIS (CHANGE 5)
# ========================================================================

print("\n" + "=" * 70)
print("STEP 11: SHAP ANALYSIS - DISCOUNT IMPACT")
print("=" * 70)

print("\n🔍 Computing SHAP values...")
print("This may take 2-5 minutes for accurate results...\n")

# Sample for SHAP (use subset for speed)
X_train_sample = X_train.sample(min(1000, len(X_train)), random_state=42)
X_test_sample = X_test.sample(min(500, len(X_test)), random_state=42)

# Create SHAP explainer
explainer = shap.TreeExplainer(xgb_model)
shap_values_train = explainer.shap_values(X_train_sample)
shap_values_test = explainer.shap_values(X_test_sample)

print("✓ SHAP values computed\n")

# SHAP summary
print("━" * 70)
print("SHAP ANALYSIS: FEATURE IMPACT ON PREDICTIONS")
print("━" * 70)

# Get mean absolute SHAP values
mean_shap = pd.DataFrame({
    'feature': X_train.columns,
    'mean_abs_shap': np.abs(shap_values_test).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print("\nTop 15 Features by SHAP Impact:\n")
print(mean_shap.head(15).to_string(index=False))

# Discount-specific SHAP analysis
discount_shap = mean_shap[mean_shap['feature'].isin(['discount_pct_clean', 'is_on_sale'])]

print("\n" + "=" * 70)
print("Q2: HOW DO DISCOUNTS AFFECT REVENUE? (SHAP INSIGHTS)")
print("=" * 70)

print("\nDiscount Feature SHAP Impact:\n")
print(discount_shap.to_string(index=False))

# Analyze discount effect by genre
print("\n" + "-" * 70)
print("DISCOUNT EFFECT BY GENRE (SHAP Analysis):")
print("-" * 70)

# Get discount_pct feature index
discount_idx = list(X_train.columns).index('discount_pct_clean')

# Calculate mean SHAP value for discount by genre
test_sample_with_genre = X_test_sample.copy()
test_sample_genres = test_sample_with_genre[genre_cols].apply(
    lambda row: get_genre_from_onehot(row, genre_cols), axis=1
)

discount_shap_by_genre = pd.DataFrame({
    'genre': test_sample_genres,
    'discount_shap': shap_values_test[:, discount_idx]
})

genre_discount_effect = discount_shap_by_genre.groupby('genre')['discount_shap'].agg(['mean', 'std', 'count'])
genre_discount_effect = genre_discount_effect[genre_discount_effect['count'] >= 10]  # Min 10 samples
genre_discount_effect = genre_discount_effect.sort_values('mean', ascending=False)

print("\nGenre-Specific Discount SHAP Values:\n")
print(genre_discount_effect.to_string())

print("\n💡 INTERPRETATION:")
print("   • Positive SHAP → Discount INCREASES predicted revenue")
print("   • Negative SHAP → Discount DECREASES predicted revenue")
print("   • Zero SHAP → Discount has no effect")

top_positive_genre = genre_discount_effect.index[0]
top_negative_genre = genre_discount_effect.index[-1]

print(f"\n   Best for discounts: {top_positive_genre.upper()}")
print(f"   → Discounts boost predicted revenue by {genre_discount_effect.loc[top_positive_genre, 'mean']:.3f} (log scale)")

print(f"\n   Worst for discounts: {top_negative_genre.upper()}")
print(f"   → Discounts hurt predicted revenue by {genre_discount_effect.loc[top_negative_genre, 'mean']:.3f} (log scale)")


STEP 11: SHAP ANALYSIS - DISCOUNT IMPACT

🔍 Computing SHAP values...
This may take 2-5 minutes for accurate results...

✓ SHAP values computed

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SHAP ANALYSIS: FEATURE IMPACT ON PREDICTIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Top 15 Features by SHAP Impact:

           feature  mean_abs_shap
       final_price       1.066986
           has_dlc       0.877114
discount_pct_clean       0.648763
       mac_support       0.279221
     dlc_available       0.192627
      genre_casual       0.150477
    multi_platform       0.068705
     linux_support       0.048220
   genre_adventure       0.047322
      genre_action       0.043348
       genre_indie       0.029771
         genre_rpg       0.027481
  genre_simulation       0.026071
   is_early_access       0.014349
    genre_strategy       0.012656

Q2: HOW DO DISCOUNTS AFFECT REVENUE? (SHAP INSIGHTS)

Discount Feature SHAP Impact:

    

In [26]:
# ========================================================================
# 12. VISUALIZATIONS
# ========================================================================

print("\n" + "=" * 70)
print("STEP 12: CREATING VISUALIZATIONS")
print("=" * 70)

sns.set(style="whitegrid", palette="muted")

# Visualization 1: Feature Importance
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance.head(15).sort_values('importance')
ax.barh(top_features['feature'], top_features['importance'], color='steelblue')
ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Feature Importance (Tuned XGBoost)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('xgboost_feature_importance_tuned.png', dpi=300, bbox_inches='tight')
print("✓ Saved: xgboost_feature_importance_tuned.png")
plt.close()

# Visualization 2: Actual vs Predicted
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(y_test_original/1e6, y_pred_test_original/1e6, alpha=0.3, s=20)
max_val = max(y_test_original.max(), y_pred_test_original.max()) / 1e6
ax.plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect Prediction')
ax.set_xlabel('Actual Revenue (₹ Millions)', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Revenue (₹ Millions)', fontsize=12, fontweight='bold')
ax.set_title(f'Tuned XGBoost: Actual vs Predicted (R² = {tuned_r2_test:.3f})',
             fontsize=14, fontweight='bold', pad=20)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('xgboost_actual_vs_predicted_tuned.png', dpi=300, bbox_inches='tight')
print("✓ Saved: xgboost_actual_vs_predicted_tuned.png")
plt.close()

# Visualization 3: Genre RMSE
fig, ax = plt.subplots(figsize=(10, 8))
top_rmse_genres = genre_rmse_sorted.head(15)
ax.barh(range(len(top_rmse_genres)), top_rmse_genres['RMSE_millions'], color='coral')
ax.set_yticks(range(len(top_rmse_genres)))
ax.set_yticklabels(top_rmse_genres.index)
ax.set_xlabel('RMSE (₹ Millions)', fontsize=12, fontweight='bold')
ax.set_ylabel('Genre', fontsize=12, fontweight='bold')
ax.set_title('Prediction Difficulty by Genre (RMSE)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('genre_rmse_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: genre_rmse_analysis.png")
plt.close()

# Visualization 4: SHAP Summary Plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_test, X_test_sample, plot_type="bar", show=False)
plt.title('SHAP Feature Importance', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_summary_plot.png', dpi=300, bbox_inches='tight')
print("✓ Saved: shap_summary_plot.png")
plt.close()

# Visualization 5: SHAP Discount Analysis
fig, ax = plt.subplots(figsize=(10, 6))
genre_discount_effect_sorted = genre_discount_effect.sort_values('mean')
colors = ['red' if x < 0 else 'green' for x in genre_discount_effect_sorted['mean']]
ax.barh(range(len(genre_discount_effect_sorted)), genre_discount_effect_sorted['mean'], color=colors)
ax.set_yticks(range(len(genre_discount_effect_sorted)))
ax.set_yticklabels(genre_discount_effect_sorted.index)
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Mean SHAP Value (Discount Effect)', fontsize=12, fontweight='bold')
ax.set_ylabel('Genre', fontsize=12, fontweight='bold')
ax.set_title('Discount Impact by Genre (SHAP Analysis)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_discount_by_genre.png', dpi=300, bbox_inches='tight')
print("✓ Saved: shap_discount_by_genre.png")
plt.close()


STEP 12: CREATING VISUALIZATIONS
✓ Saved: xgboost_feature_importance_tuned.png
✓ Saved: xgboost_actual_vs_predicted_tuned.png
✓ Saved: genre_rmse_analysis.png
✓ Saved: shap_summary_plot.png
✓ Saved: shap_discount_by_genre.png


In [27]:
# ========================================================================
# 13. SAVE RESULTS
# ========================================================================

print("\n" + "=" * 70)
print("STEP 13: SAVING RESULTS")
print("=" * 70)

# Save feature importance
feature_importance.to_csv('xgboost_feature_importance_tuned.csv', index=False)
print("✓ Saved: xgboost_feature_importance_tuned.csv")

# Save predictions
predictions_df = pd.DataFrame({
    'actual_revenue': y_test_original,
    'predicted_revenue': y_pred_test_original,
    'error': y_test_original - y_pred_test_original,
    'error_pct': ((y_test_original - y_pred_test_original) / (y_test_original + 1) * 100),
    'genre': test_genres.values
})
predictions_df.to_csv('xgboost_predictions_tuned.csv', index=False)
print("✓ Saved: xgboost_predictions_tuned.csv")

# Save genre RMSE analysis
genre_rmse_sorted.to_csv('genre_rmse_analysis.csv')
print("✓ Saved: genre_rmse_analysis.csv")

# Save SHAP results
genre_discount_effect.to_csv('shap_discount_by_genre.csv')
print("✓ Saved: shap_discount_by_genre.csv")

# Save model
import joblib
joblib.dump(xgb_model, 'xgboost_revenue_model_tuned.pkl')
print("✓ Saved: xgboost_revenue_model_tuned.pkl")

joblib.dump(scaler, 'feature_scaler.pkl')
print("✓ Saved: feature_scaler.pkl")

# Save best parameters
import json
with open('best_hyperparameters.json', 'w') as f:
    json.dump(grid_search.best_params_, f, indent=4)
print("✓ Saved: best_hyperparameters.json")

# Save model performance summary
summary = {
    'Metric': ['Training Samples', 'Test Samples', 'Features', 'Zero Revenue Games %',
               'Ridge R²', 'XGBoost Initial R²', 'XGBoost Tuned R²',
               'Test RMSE (₹M)', 'Test MAE (₹M)', 'Train-Test Gap'],
    'Value': [len(X_train), len(X_test), X_train.shape[1], f'{zero_revenue_count/len(df_model)*100:.1f}%',
              f'{baseline_r2:.4f}', f'{initial_r2:.4f}', f'{tuned_r2_test:.4f}',
              f'{tuned_rmse_original/1e6:.2f}', f'{tuned_mae_original/1e6:.2f}',
              f'{tuned_r2_train - tuned_r2_test:.4f}']
}
summary_df = pd.DataFrame(summary)
summary_df.to_csv('model_performance_summary_tuned.csv', index=False)
print("✓ Saved: model_performance_summary_tuned.csv")

print("\n" + "=" * 70)
print("✓ MODEL TRAINING COMPLETE!")
print("=" * 70)
print("\nNext steps:")
print("  1. Review SHAP analysis for discount insights")
print("  2. Build Flask/FastAPI server (next script)")
print("  3. Create GitHub repository with README")
print("  4. Try other models (ANN, LightGBM, etc.)")


STEP 13: SAVING RESULTS
✓ Saved: xgboost_feature_importance_tuned.csv
✓ Saved: xgboost_predictions_tuned.csv
✓ Saved: genre_rmse_analysis.csv
✓ Saved: shap_discount_by_genre.csv
✓ Saved: xgboost_revenue_model_tuned.pkl
✓ Saved: feature_scaler.pkl
✓ Saved: best_hyperparameters.json
✓ Saved: model_performance_summary_tuned.csv

✓ MODEL TRAINING COMPLETE!

Next steps:
  1. Review SHAP analysis for discount insights
  2. Build Flask/FastAPI server (next script)
  3. Create GitHub repository with README
  4. Try other models (ANN, LightGBM, etc.)
